# 06 · Model Configuration: `ConfigDict`, Strict Mode, Extra Fields, Immutability & Private Attributes

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 45 min

### 🎯 What you will learn
- `model_config = ConfigDict(...)` and the most important options
- **Extra fields**: `ignore` (default), `forbid`, `allow`
- **Strict mode** (no coercion) for a model, a field, or a single call
- `validate_assignment`, `frozen` (immutable & hashable models), `revalidate_instances`
- String handling: `str_strip_whitespace`, `str_to_lower`, `str_max_length`
- `from_attributes` (validate ORM/arbitrary objects), `use_enum_values`, `arbitrary_types_allowed`
- `PrivateAttr` and `ClassVar`
- ⚠️ v1's `class Config:` vs v2's `model_config`

In [1]:
from enum import Enum
from typing import ClassVar
from pydantic import BaseModel, ConfigDict, Field, ValidationError, PrivateAttr

def attempt(fn):
    try:
        print("✅", fn())
    except ValidationError as e:
        print("❌", "; ".join(f"{'.'.join(map(str, x['loc']))}: {x['msg']}" for x in e.errors()))

## 1. Extra fields

In [2]:
data = {"name": "Asha", "age": 31, "is_admin": True}          # an unexpected key

class Ignore(BaseModel):                                      # default: silently drop extras
    name: str
    age: int

class Forbid(BaseModel):
    model_config = ConfigDict(extra="forbid")                 # ✅ recommended for API inputs
    name: str
    age: int

class Allow(BaseModel):
    model_config = ConfigDict(extra="allow")                  # keep them
    name: str
    age: int

attempt(lambda: Ignore(**data))
attempt(lambda: Forbid(**data))
a = Allow(**data)
print("✅", a, "| extras:", a.model_extra)

✅ name='Asha' age=31
❌ is_admin: Extra inputs are not permitted
✅ name='Asha' age=31 is_admin=True | extras: {'is_admin': True}


`extra="forbid"` catches typos (`"emial"`) and mass-assignment attacks (a client sending `"is_admin": true`).

## 2. Strict mode

Lax mode converts `"42"` → `42`. **Strict** mode requires the exact type (JSON input still allows the natural JSON types, e.g. strings for dates):

In [3]:
class LaxPoint(BaseModel):
    x: int
    y: int

class StrictPoint(BaseModel):
    model_config = ConfigDict(strict=True)
    x: int
    y: int

class MixedPoint(BaseModel):
    x: int = Field(strict=True)             # strict for ONE field
    y: int

attempt(lambda: LaxPoint(x="1", y=2.0))
attempt(lambda: StrictPoint(x="1", y=2.0))
attempt(lambda: MixedPoint(x="1", y="2"))
attempt(lambda: LaxPoint.model_validate({"x": "1", "y": "2"}, strict=True))    # strict for one call

✅ x=1 y=2
❌ x: Input should be a valid integer; y: Input should be a valid integer
❌ x: Input should be a valid integer
❌ x: Input should be a valid integer; y: Input should be a valid integer


Also available: `from pydantic import StrictInt, StrictStr, StrictBool` as field types.

## 3. Validate on assignment & frozen models

In [4]:
class Account(BaseModel):
    model_config = ConfigDict(validate_assignment=True)
    owner: str
    balance: float = Field(ge=0)

acct = Account(owner="Asha", balance=100)
acct.balance = "250.5"                  # validated & coerced
print(acct.balance)
try:
    acct.balance = -5
except ValidationError as e:
    print("❌", e.errors()[0]["msg"])

class Coordinate(BaseModel):
    model_config = ConfigDict(frozen=True)        # immutable → also hashable
    lat: float
    lon: float

pune = Coordinate(lat=18.52, lon=73.86)
try:
    pune.lat = 0
except ValidationError as e:
    print("❌", e.errors()[0]["msg"])
print({pune: "Pune"}[Coordinate(lat=18.52, lon=73.86)])        # usable as a dict key
print(pune.model_copy(update={"lat": 19.0}))                    # create a modified copy instead

250.5
❌ Input should be greater than or equal to 0
❌ Instance is frozen
Pune
lat=19.0 lon=73.86


## 4. String options

In [5]:
class Clean(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, str_to_lower=True, str_max_length=20)
    username: str
    city: str

attempt(lambda: Clean(username="  Asha_R ", city="  PUNE"))
attempt(lambda: Clean(username="x" * 30, city="ok"))

✅ username='asha_r' city='pune'
❌ username: String should have at most 20 characters


## 5. `from_attributes`: validate objects, not just dicts

Essential for ORMs (SQLAlchemy, Django): build a Pydantic model **from an object's attributes**.

In [6]:
class UserRow:                                   # imagine a SQLAlchemy ORM object
    def __init__(self, id, email, password_hash):
        self.id, self.email, self.password_hash = id, email, password_hash

class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)      # ⚠️ v1 name: orm_mode = True
    id: int
    email: str                                          # password_hash is NOT exposed

print(UserOut.model_validate(UserRow(1, "asha@example.com", "$2b$12$...")))

id=1 email='asha@example.com'


## 6. Enums, arbitrary types and other options

In [7]:
class Status(str, Enum):
    ACTIVE = "active"
    BANNED = "banned"

class WithEnum(BaseModel):
    model_config = ConfigDict(use_enum_values=True)     # store "active" instead of Status.ACTIVE
    status: Status

print(WithEnum(status="active").status, type(WithEnum(status="active").status).__name__)

class Connection:                                      # a class Pydantic doesn't know
    def __init__(self, dsn): self.dsn = dsn

class Service(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)    # validated only with isinstance()
    name: str
    conn: Connection

print(Service(name="db", conn=Connection("sqlite://")).conn.dsn)

active str
sqlite://


| Option | Effect |
|---|---|
| `extra` | `"ignore"` / `"forbid"` / `"allow"` |
| `strict` | no type coercion |
| `frozen` | immutable + hashable |
| `validate_assignment` | validate `obj.x = v` |
| `validate_default` | validate default values |
| `from_attributes` | accept objects (ORM mode) |
| `alias_generator`, `validate_by_name`, `validate_by_alias`, `serialize_by_alias` | aliases (lesson 04) |
| `str_strip_whitespace`, `str_to_lower/upper`, `str_min/max_length` | string handling |
| `use_enum_values` | store enum values |
| `arbitrary_types_allowed` | allow non-Pydantic classes (isinstance check) |
| `revalidate_instances` | `"never"` / `"always"` / `"subclass-instances"`: re-validate model instances passed as fields |
| `ser_json_timedelta`, `ser_json_bytes` | JSON formats for those types |
| `title`, `json_schema_extra` | JSON Schema |
| `use_attribute_docstrings` | field docstrings become descriptions |
| `defer_build` | build validators lazily (faster import for huge apps) |

Configuration is **inherited**: put shared settings on a base model.

In [8]:
class StrictAPIModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True, frozen=True)

class CreateTag(StrictAPIModel):
    name: str

attempt(lambda: CreateTag(name=" news ", color="red"))
print(CreateTag(name=" news ").name)

❌ color: Extra inputs are not permitted
news


## 7. Private attributes and class variables

Fields are validated and serialised. Sometimes you need **internal state** that isn't:

In [9]:
from datetime import datetime

class Cart(BaseModel):
    TAX_RATE: ClassVar[float] = 0.18                  # class-level constant, not a field
    items: dict[str, int] = {}
    _created: datetime = PrivateAttr(default_factory=datetime.now)
    _cache: dict = PrivateAttr(default_factory=dict)

    def total(self, prices: dict[str, float]) -> float:
        if "total" not in self._cache:
            self._cache["total"] = sum(prices[k] * q for k, q in self.items.items()) * (1 + self.TAX_RATE)
        return round(self._cache["total"], 2)

cart = Cart(items={"pen": 2})
print(cart.total({"pen": 10}), cart.model_dump(), list(Cart.model_fields), type(cart._created).__name__)

23.6 {'items': {'pen': 2}} ['items'] datetime


Attributes starting with `_` are private (not fields); `ClassVar` annotations are class attributes.

## 8. ⚠️ v1 configuration style

```python
class User(BaseModel):          # Pydantic v1 (deprecated in v2)
    class Config:
        orm_mode = True
        allow_population_by_field_name = True
        anystr_strip_whitespace = True
        extra = "forbid"
```
```python
class User(BaseModel):          # ✅ Pydantic v2
    model_config = ConfigDict(from_attributes=True, validate_by_name=True,
                              str_strip_whitespace=True, extra="forbid")
```
(Full migration table in lesson 11.)

## 📝 Summary
- Configure models with `model_config = ConfigDict(...)`, inherited by subclasses.
- `extra="forbid"` for inputs; `strict=True` (model, field or call) to disable coercion.
- `validate_assignment`, `frozen` (immutable, hashable), `from_attributes` (ORM objects).
- `PrivateAttr` for internal state, `ClassVar` for constants.

## 🏋️ Practice
1. Create a base `InputModel` (`extra="forbid"`, `str_strip_whitespace=True`) and two subclasses; show that typos in keys are rejected.
2. Make a frozen `Money(amount: Decimal, currency: str)` model and use instances as keys in a dict of exchange rates.
3. Validate a SQLAlchemy object from Module 14 · 06 into a Pydantic model with `from_attributes=True`.